**Problem 2**: We have established that our method only works with linear functions. Suppose we have higher order curves, or sinusoids, how can regression or machine learning discover the underlying correlation?

Because our methods have been linear, if we plug in data for a quadratic, for example, the machine will try its best to find a line that best follows the given dataset. But well, that's not we want. Classically, we'll have to guess. Suppose we know that our function is a polynomial of the form
$$ f(x) = x^7 + 3x^5 + 18x^2 +1$$
Now let's make an observation: if we let the coefficients before each term to be weights $w_i$, we can see that:
$$f(k w_i) = k f(w_i)$$
meaning, our function is linear in its parameters. This means we can still use linear regression. $x^7, \ldots$ are now called basis functions for our regression. We can still write $\mathbf{y} = X \theta$, but $X$ now contains our basis functions:
$$X = \begin{bmatrix}
1 & x_1 & \ldots & x_2^7 \\
1 & x_2 & \ddots & x_2^7 \\
\vdots & \vdots & \vdots & \vdots 
\end{bmatrix}  $$
And we can solve for the normal equations like normal. Under the hood, there are tools like numpy that do this very well:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

x = np.linspace(-2, 2, 200)

y_true = x**7 + 3*x**5 + 18*x**2 + 1
noise = np.random.normal(0, 2, size=x.shape)
y = y_true + noise

coeffs = np.polyfit(x, y, deg=7)

print("Recovered coefficients:")
print(coeffs)

y_pred = np.polyval(coeffs, x)

plt.figure(figsize=(8,5))
plt.scatter(x, y, s=10, alpha=0.4, label="Noisy data")
plt.plot(x, y_true, linewidth=3, label="True function")
plt.plot(x, y_pred, "--", linewidth=2, label="Recovered polynomial")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)

# -----------------------
# Generate synthetic data
# -----------------------

x = torch.linspace(-2, 2, 200).unsqueeze(1)

y_true = (x**7 + 3*x**5 + 18*x**2 + 1)

noise = 2 * torch.randn_like(y_true)
y = y_true + noise


class Polynomial(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.coeffs = torch.nn.Parameter(torch.randn(8, 1))

    def forward(self, x):

        powers = torch.cat(
            [x**7,
             x**6,
             x**5,
             x**4,
             x**3,
             x**2,
             x,
             torch.ones_like(x)],
            dim=1
        )

        return powers @ self.coeffs


model = Polynomial()

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)


for epoch in range(5000):

    optimizer.zero_grad()

    pred = model(x)

    loss = criterion(pred, y)

    loss.backward()

    optimizer.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())

print("\nRecovered coefficients:")

for c in model.coeffs.detach().flatten():
    print(c.item())

with torch.no_grad():
    y_pred = model(x)

plt.figure(figsize=(8,5))
plt.scatter(x.numpy(), y.numpy(), s=10, alpha=0.4, label="Noisy data")
plt.plot(x.numpy(), y_true.numpy(), linewidth=3, label="True")
plt.plot(x.numpy(), y_pred.numpy(), "--", linewidth=2, label="PyTorch")
plt.legend()
plt.show()

So under the hood, both methods still work the same: they still use MSE, and given enough time, the numerical optimisations in our model will converge very closely to the exact analytical solution given by numpy. Notice how instead of gradient descent, we used the Adam optimiser.

Instead of solving the normal equations directly, our PyTorch model treats the polynomial coefficients as trainable parameters and updates them iteratively using the **Adam** optimizer (**Adaptive Moment Estimation**).

Recall that in gradient descent, we repeatedly update the parameters according to

$$
\theta \leftarrow \theta - \alpha \nabla_\theta L,
$$

where $\alpha$ is the learning rate and $\nabla_\theta L$ is the gradient of the loss.

Adam improves upon this by maintaining two running estimates for every parameter:

- The **first moment** (an exponentially weighted average of past gradients), which acts like momentum.
- The **second moment** (an exponentially weighted average of the squared gradients), which estimates how large or noisy the gradients are.

The update equations are

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t,
$$

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2,
$$

where $g_t$ is the current gradient. Adam then updates the parameters using

$$
\theta \leftarrow
\theta -
\alpha
\frac{m_t}{\sqrt{v_t}+\varepsilon}.
$$

Intuitively, the momentum term smooths out noisy updates by averaging gradients over time, while the adaptive scaling reduces the learning rate for parameters that consistently receive large gradients and increases it for those with small gradients. This makes optimization more stable and often requires much less manual tuning than vanilla gradient descent.

### Adam vs. SGD

For this polynomial regression problem, Adam generally converges faster than vanilla Stochastic Gradient Descent (SGD). The polynomial basis functions have very different scales—for example, $x$ ranges roughly from $[-2,2]$, while $x^7$ ranges from $[-128,128]$. Consequently, the corresponding gradients also have very different magnitudes. Adam automatically adapts the step size for each coefficient, allowing all parameters to learn at a reasonable pace, whereas SGD uses the same learning rate for every parameter and may require careful tuning to converge efficiently.

However, Adam is **not** the best optimizer for every problem. In this example, the model is still linear in its parameters, so an exact least-squares solution exists. Methods such as NumPy's `polyfit` (or `torch.linalg.lstsq`) recover the optimal coefficients directly using linear algebra and are both faster and more accurate than any iterative optimizer. In large neural networks where no closed-form solution exists, Adam is often an excellent default choice because of its fast convergence and robustness to noisy gradients. Nevertheless, for very large-scale deep learning, SGD with momentum sometimes achieves better final generalization performance after careful tuning, even though it usually converges more slowly.

But there are cases where classical linear regression fails. We have established that our functions need to be linear in parameters. But for example, we're trying to recover a signal made of sinusoids. If we know certain frequencies and we know that their amplitudes are large enough such that other frequencies present are noise, we can choose a different basis:
$$A\sin(\omega _1 x), \hspace{10pt} B\cos(\omega _2 x), \ldots$$

Now we can still solve for $A$ and $B$, or other amplitudes because it is linear. Our bases are now simply $\sin(\omega _1 x)$ and $\cos(\omega _2 x)$. In fact, for any combination of functions that are linear in their parameters, we can construct a set of basis functions and solve for the general
$$f(x) = \sum _i w_i \phi _i$$
using least squares, where $\phi _i$ are the basis functions. But there are functions like $\frac{1}{ax +b}$ or $e^{-ax}+c$ where this is not possible.  
So say we want to learn $\sin(3x + \frac{\pi}{4}) + 4cos(8x-\frac{\pi}{8})$, not knowing both the amplitudes and the frequencies. How are we going to do this?

A classical way to do this is by using the Fast Fourier transform. Basically, we know that the continuous Fourier transform turns a signal $x(t)$ into a sum of sinusoids: almost exactly what we need here. In the real world, we don't have continuous signals. Computers always receive a discrete analog signal that when interpolated, resembles a signal. So we need the Discrete Fourier transform to do that for discrete sampled data. The Fast Fourier transform, arguably the greatest algorithm of the 20th century, gives us a way to calculate the DFT quickly. Using numpy, we can implement this like so:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 1024
x = np.linspace(0, 2*np.pi, N, endpoint=False)

noise = np.random.normal(0,0.5)

y = (
    np.sin(3*x + np.pi/4)
    + 4*np.cos(8*x - np.pi/8)
) + noise

Y = np.fft.rfft(y)

freq = np.fft.rfftfreq(N, d=(x[1] - x[0]))
omega = 2*np.pi*freq

amplitude = 2*np.abs(Y) / N
phase = np.angle(Y)

plt.figure(figsize=(10,4))

plt.stem(omega, amplitude)

plt.xlim(0, 15)

plt.xlabel("Angular Frequency")
plt.ylabel("Amplitude")
plt.title("FFT Magnitude Spectrum")

plt.show()

threshold = 0.1

print("Recovered components:\n")

for w, A, phi in zip(omega, amplitude, phase):

    if A > threshold:

        print(f"ω = {w:.3f}, amplitude = {A:.3f}, phase = {phi:.3f}")

Notice that we recover an exact signal, up to noise. The frequency-0 component of the FFT corresponds to the average (DC) value of the signal. A nonzero mean—whether from the signal itself or from finite-sample noise—appears as a spike at zero frequency. It is common practice to subtract the mean before computing the FFT when one is interested only in the oscillatory components. 

So basically, we've moved from the linear basis for linear regression, to polynomial basis, and now to the Fourier basis, which comprises of purely sines and cosines. Choosing bases mean that we at least have some understanding of the system. But remember, our neural network doesn't. Up till now, we've only been having linear neurons. They work for linear regression for different bases, but remember, we do not know the frequency beforehand. We'll have to introduce nonlinearity into the neural network. One way to do this is through activation functions: using them as our basis functions. The premise is that we can approximate the function in the trained region pretty well using lots of combinations of these activation functions. There are lots of activation functions, each serving a different purpose. For this example, we're going to use $\tanh(x)$. It's smooth, it's reliable, and people had been relying on it in early ML.

Our architecture now looks like this:
```mermaid
flowchart LR
    A["(N, 1)<br/>Input"] --> B["Linear<br/>1×128"]
    B --> C["(N, 128)"]
    C --> D["Tanh"]
    D --> E["Linear<br/>128×128"]
    E --> F["(N, 128)"]
    F --> G["Tanh"]
    G --> H["Linear<br/>128×1"]
    H --> I["(N, 1)<br/>Prediction"]
```

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)

x = torch.linspace(
    0,
    2*torch.pi,
    500
).unsqueeze(1)

y = (
    torch.sin(3*x + torch.pi/4)
    + 4*torch.cos(8*x - torch.pi/8)
)



model = nn.Sequential(

    nn.Linear(1,128),
    nn.Tanh(),

    nn.Linear(128,128),
    nn.Tanh(),

    nn.Linear(128,1)

)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

for epoch in range(6000):

    optimizer.zero_grad()

    pred = model(x)

    loss = criterion(pred,y)

    loss.backward()

    optimizer.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())

with torch.no_grad():

    x_plot = torch.linspace(
        -2 * torch.pi,
        4 * torch.pi,
        1000
    ).unsqueeze(1)

    y_true = (
        torch.sin(3 * x_plot + torch.pi / 4)
        + 4 * torch.cos(8 * x_plot - torch.pi / 8)
    )

    y_pred = model(x_plot)

print(sum(p.numel() for p in model.parameters()))

plt.figure(figsize=(14, 5))

# True function
plt.plot(
    x_plot.numpy(),
    y_true.numpy(),
    linewidth=3,
    label="True Signal"
)

# Neural network
plt.plot(
    x_plot.numpy(),
    y_pred.numpy(),
    "--",
    linewidth=2,
    label="Neural Network"
)

# Show training region
plt.axvspan(
    0,
    2 * torch.pi,
    color="lightgreen",
    alpha=0.2,
    label="Training Region"
)

plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)

plt.show()


And you can see from the figure that the neural network fits the signal extremely well within the training interval, yet once we move outside that region its predictions rapidly diverge from the true function. In other words, it interpolates accurately but extrapolates poorly. This is not an accident. It is a consequence of one of the fundamental results in deep learning, the **Universal Approximation Theorem**. In one of its common forms, the theorem states that a feedforward neural network with a single hidden layer containing sufficiently many neurons and a suitable nonlinear activation function can approximate any continuous function on a compact (bounded and closed) subset of $\mathbb{R}^n$ to arbitrary accuracy. Modern deep networks extend this idea by using many hidden layers, often requiring far fewer neurons to achieve the same level of approximation.

It is important to note what the theorem does not say. It makes no guarantees about how many neurons are required, whether gradient-based optimization will actually find a good solution, or how the network behaves outside the region on which it was trained. The theorem concerns approximation only on the bounded domain where the data are available. Consequently, although our network reproduces the sinusoidal signal almost perfectly over the training interval, it has no reason to continue the same periodic pattern beyond it.

Finally, while neural networks are remarkably flexible function approximators, they are not always the best tool. For low-dimensional problems where we possess a good mathematical model, classical methods remain superior. Linear regression, polynomial regression, splines, Fourier analysis, and other basis-function techniques exploit known structure in the problem, producing solutions that are typically faster, more accurate, more interpretable, and often exact. Neural networks become advantageous when such structure is unknown or when the input space is extremely high-dimensional—as in computer vision, speech recognition, natural language processing, and robotics—where constructing an appropriate set of basis functions by hand is either impractical or impossible. So for our purpose of fitting data to an unknown curve from samples: **problem solved**.
